In [9]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures

import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from src import read_dataset

plt.rcParams.update({'font.size': 20})
sns.set_style("darkgrid")

# Feature Extraction
In this notebook, we're going to create features derived from the original 7.
- Combination of polynomials up to the third order
- $\dfrac{ng}{np}$ ratio
- Normalization of density altitude
- Normalization of air density

In [10]:
# 1s
df_x, df_y = read_dataset('original')

In [11]:
# 0s
# Constants for ISA model
T0 = 288.15  # Sea-level standard temperature (K)
p0 = 101325  # Sea-level standard atmospheric pressure (Pa)
a = 0.0065  # Temperature lapse rate (K/m)
g = 9.80665  # Gravitational acceleration (m/s^2)
R = 8.3144598  # Universal gas constant (J/(mol·K))
RS = 287.05  # Specific gas constant for air (J/(kg·K))

In [12]:
# 1s
new_df_x = df_x.copy()
poly = PolynomialFeatures(degree=3, include_bias=False)
to_poly = df_x[['oat', 'mgt', 'pa', 'np', 'ng']]
poly_features = poly.fit_transform(to_poly)
poly_df = pd.DataFrame(poly_features, columns=poly.get_feature_names_out(to_poly.columns))
new_df_x = pd.concat([new_df_x, poly_df], axis=1)

# Add NP/NG ratio feature
new_df_x['np_ng_ratio'] = df_x['np'] / df_x['ng']

# Air density formula: da = 1.2376 * pa + 118.8 * oat - 1782
new_df_x['da'] = (1.2376 * df_x['pa']) + (118.8 * (df_x['oat'] + 273.15)) - 1782

# Compute pressure at altitude (h in meters, assuming pressure altitude in dataset is in feet)
new_df_x['h_m'] = df_x['pa'] * 0.3048  # Convert pressure altitude from feet to meters
new_df_x['P'] = p0 * (1 - (a * new_df_x['h_m']) / T0) ** (g / (R * a))

# Compute air density (rho), ensuring valid values
new_df_x['rho'] = new_df_x['P'] / (RS * (df_x['oat'] + 273.15))

# Normalize original features based on air density (rho)
for col in df_x.columns:
    new_df_x[f'{col}_norm_da'] = df_x[col] / new_df_x['da']
    new_df_x[f'{col}_air_density'] = df_x[col] / new_df_x['rho']
new_df_x.head()

,trq_measured,oat,mgt,pa,ias,np,ng,oat,mgt,pa,...,mgt_norm_da,mgt_air_density,pa_norm_da,pa_air_density,ias_norm_da,ias_air_density,np_norm_da,np_air_density,ng_norm_da,ng_air_density
0,54.100,2.00000,544.5000,212.1408,74.56250,89.18000,99.6400,2.00000,544.5000,212.1408,...,0.017470,553.143257,0.006806,215.508270,0.002392,75.746087,0.002861,90.595621,0.003197,101.221661
1,49.625,24.22231,578.4844,1625.6400,30.35596,99.55273,91.3866,24.22231,578.4844,1625.6400,...,0.016269,3746.466410,0.045718,10528.210709,0.000854,196.595767,0.002800,644.738145,0.002570,591.851444
2,52.000,7.00000,566.1000,1912.9250,65.62500,100.14000,90.9600,7.00000,566.1000,1912.9250,...,0.016715,4964.683468,0.056483,16776.306524,0.001938,575.529681,0.002957,878.225406,0.002686,797.717026
3,62.400,7.25000,560.1000,277.0632,54.81250,90.64000,100.2800,7.25000,560.1000,277.0632,...,0.017573,628.854701,0.008693,311.073908,0.001720,61.540972,0.002844,101.766453,0.003146,112.589804
4,62.900,23.25000,593.7000,53.6448,73.43750,99.91000,92.1700,23.25000,593.7000,53.6448,...,0.017724,533.038029,0.001601,48.163582,0.002192,65.933940,0.002983,89.701582,0.002752,82.752426


## Export to `.csv` files

In [14]:
# 60s
os.makedirs('../dataset/1-preprocessed', exist_ok=True)
new_df_x.to_csv('../dataset/1-preprocessed/X.csv', index=False)
df_y.to_csv('../dataset/1-preprocessed/y.csv', index=False)